```yaml
title: Mapping a NAICS code to its real federal buyers
description: Given a NAICS code, rank the agencies actually spending money on it — with a year-over-year delta to spot risers and fallers.
tags: [contracts, filtering, shape, joining, business-development]
endpoints: [list_contracts, list_naics]
```

# Mapping a NAICS code to its real federal buyers

A BD lead picks NAICS codes from their company's SAM registration and asks: *where should we be selling?* The honest answer isn't on any agency website. It's distributed across thousands of SAM records. USASpending is a good starting point. But, let's see what Tango can do using the `list_contracts` method.

## Setup

In [1]:
from collections import defaultdict
from tango import TangoClient

client = TangoClient()  # reads TANGO_API_KEY from env

## Pick a NAICS

We'll use **541512 — Computer Systems Design Services**, the workhorse NAICS for federal IT integrators. Tango's `list_naics` confirms the code and its small-business size standard.

In [2]:
NAICS = "541512"

r = client.list_naics(search=NAICS, limit=5, shape="code,description,size_standards(*)")
for n in r.results:
    if str(n.get("code")) == NAICS:
        rev = n["size_standards"]["revenue_limit"]
        emp = n["size_standards"]["employee_limit"]
        size = []
        if rev is not None:
            size.append(f"revenue ≤ ${rev/1e6}M")
        if emp is not None:
            size.append(f"employees ≤ {emp}")
        print(f"{n['code']}  {n.get('description')}")
        if size:
            print(f"  size standard: {'  |  '.join(size)}")
        break


541512  Computer Systems Design Services
  size standard: revenue ≤ $34.0M


## Pull the mid-band contracts under that NAICS

`list_contracts` paginates by cursor, so we walk the cursor and stop when it's gone. We filter to obligations between **$1M and $5M** for two reasons. The floor (`obligated_gte=$1M`) cuts out task-order noise and mods that won't change the agency ranking. The ceiling (`obligated_lte=$5M`) cuts out the mega-IDIQs and ceiling actions — they're individually huge and would drown out the steady, addressable mid-band where a typical BD pursuit actually plays. We also `shape` the response to just the fields we'll aggregate on (the awarding office, obligated dollars, fiscal year) so each page stays small.

In [3]:
SHAPE = (
    "key,obligated,fiscal_year,description,"
    "awarding_office(*),set_aside,competition,"
    "recipient"
)
MIN_OBLIGATED = "1_000_000"  # $1M floor — cuts task-order noise
MAX_OBLIGATED = "5_000_000"  # $5M ceiling — cuts mega-IDIQs that distort the ranking
MAX_PAGES = 6  # cap at ~600 contracts per year so this notebook stays CI-friendly


def pull_year(fiscal_year):
    out, cursor = [], None
    for _ in range(MAX_PAGES):
        r = client.list_contracts(
            naics_code=NAICS,
            fiscal_year=fiscal_year,
            obligated_gte=MIN_OBLIGATED,
            obligated_lte=MAX_OBLIGATED,
            shape=SHAPE,
            limit=100,
            cursor=cursor,
        )
        out.extend(r.results)
        cursor = r.cursor
        if not cursor:
            break
    return out


fy25 = pull_year(2025)
print(f"FY25: pulled {len(fy25)} contracts under NAICS {NAICS} with between $1M and $5M obligated.")

FY25: pulled 600 contracts under NAICS 541512 with between $1M and $5M obligated.


In [4]:
# Eyeball the first three results

for k in fy25[:3]:
    print(f"KEY: {k.key}  DESCRIPTION: {k.description}  OBLIGATION: {k.obligated}")

KEY: CONT_AWD_75P00125F80140_7570_47QTCA18D00L8_4732  DESCRIPTION: CYBERSECURITY OPERATIONS REPORTING AND ANALYTIC SUPPORT SERVICES (CORA)  OBLIGATION: 2050420.51
KEY: CONT_AWD_70RCSJ25FR0000045_7001_47QTCH18D0013_4732  DESCRIPTION: TECHNOLOGY OPERATIONS CENTER (TOC) SUPPORT  OBLIGATION: 4000752.9400000004
KEY: CONT_AWD_2031JW25F00085_2046_2031JW25A00002_2046  DESCRIPTION: SERVICENOW BPA ORDER - EPIC TEAM SERVICES  OBLIGATION: 1151586.0


## Roll up by department

Departments are the right grain for a target list — they're who you build relationships with. Sub-agencies and offices come later, after you've picked the door to knock on.

In [5]:
def by_department(contracts):
    agg = defaultdict(lambda: {"obligated": 0.0, "count": 0})
    for c in contracts:
        office = c.get("awarding_office") or {}
        dept = office.get("department_name") or office.get("agency_name") or "(unknown)"
        agg[dept]["obligated"] += float(c.get("obligated") or 0)
        agg[dept]["count"] += 1
    return agg


ranked = sorted(by_department(fy25).items(), key=lambda x: x[1]["obligated"], reverse=True)

print(f"Top 10 buyers of NAICS {NAICS} in FY25 (between $1M and $5M deals)\n")
print(f"  {'Dollars':>10}  {'Deals':>6}  Department")
for dept, v in ranked[:10]:
    print(f"  ${v['obligated']/1e6:>8.1f}M  {v['count']:>6}  {dept}")

Top 10 buyers of NAICS 541512 in FY25 (between $1M and $5M deals)

     Dollars   Deals  Department
  $   781.3M     326  DEPT OF DEFENSE
  $   123.6M      53  HEALTH AND HUMAN SERVICES, DEPARTMENT OF
  $   114.4M      45  HOMELAND SECURITY, DEPARTMENT OF
  $    86.6M      35  TREASURY, DEPARTMENT OF THE
  $    59.7M      30  TRANSPORTATION, DEPARTMENT OF
  $    40.2M      14  COMMERCE, DEPARTMENT OF
  $    30.8M      12  GENERAL SERVICES ADMINISTRATION
  $    30.5M       9  STATE, DEPARTMENT OF
  $    28.2M      11  AGRICULTURE, DEPARTMENT OF
  $    28.0M      12  INTERIOR, DEPARTMENT OF THE


## Roll up by set-aside

Department tells you *who* is buying. Set-aside tells you *who's allowed to sell to them*. Bucketing the same dollars by `set_aside` reveals how much of this NAICS, at this dollar band, flows through small-business programs vs. full-and-open competition. If you're an 8(a) or SDVOSB, this rollup is the one that decides whether the agency-level ranking matters to you at all.


In [6]:
def label(value, fallback="(none)"):
    """set_aside / extent_competed sometimes come back as a {code, description}
    dict and sometimes as a bare string, depending on shape resolution. Coerce."""
    if isinstance(value, dict):
        return value.get("description") or value.get("code") or fallback
    return value or fallback


def roll_up(contracts, field):
    agg = defaultdict(lambda: {"obligated": 0.0, "count": 0})
    for c in contracts:
        key = label(c.get(field))
        agg[key]["obligated"] += float(c.get("obligated") or 0)
        agg[key]["count"] += 1
    return agg


def print_table(agg, *, label_header, top=10):
    total = sum(v["obligated"] for v in agg.values()) or 1.0
    rows = sorted(agg.items(), key=lambda x: -x[1]["obligated"])
    print(f"  {'Dollars':>10}  {'Share':>6}  {'Deals':>6}  {label_header}")
    for k, v in rows[:top]:
        print(f"  ${v['obligated']/1e6:>8.1f}M  {v['obligated']/total:>5.0%}  {v['count']:>6}  {k}")


print(f"FY25 NAICS {NAICS} — by set-aside type\n")
print_table(roll_up(fy25, "set_aside"), label_header="Set-aside")


FY25 NAICS 541512 — by set-aside type

     Dollars   Share   Deals  Set-aside
  $  1086.9M    75%     452  (none)
  $   192.2M    13%      79  8AN
  $    74.3M     5%      32  8A
  $    65.4M     5%      24  SBA
  $    18.9M     1%       7  SDVOSBC
  $     5.9M     0%       2  SDVOSBS
  $     2.2M     0%       1  HZS
  $     2.2M     0%       1  HZC
  $     1.8M     0%       1  WOSBSS
  $     1.1M     0%       1  WOSB


## Roll up by competition extent

The `competition.extent_competed` field is the cleanest signal of *how* the dollars went out the door: full and open, set-aside competition, limited sources, sole source. A NAICS where most of the money moves sole-source is one where capture is about pre-positioning on incumbent vehicles — not about writing the best proposal.


In [7]:
def extent(c):
    comp = c.get("competition") or {}
    return label(comp.get("extent_competed"))


extent_agg = defaultdict(lambda: {"obligated": 0.0, "count": 0})
for c in fy25:
    k = extent(c)
    extent_agg[k]["obligated"] += float(c.get("obligated") or 0)
    extent_agg[k]["count"] += 1

print(f"FY25 NAICS {NAICS} — by competition extent\n")
print_table(extent_agg, label_header="Extent competed")


FY25 NAICS 541512 — by competition extent

     Dollars   Share   Deals  Extent competed
  $   689.2M    48%     303  Full and Open Competition
  $   362.8M    25%     145  Full and Open Competition after exclusion of sources
  $   157.8M    11%      57  Not Available for Competition
  $   157.5M    11%      62  Not Competed
  $    47.4M     3%      22  Not Competed under SAP
  $    20.7M     1%       6  (none)
  $    15.2M     1%       5  Competed under SAP


## Add a YoY lens

A static ranking tells you who's big. A delta tells you who's moving. Pull FY24 and diff — the departments where last year's number jumped (or fell off a cliff) are where the strategic narrative is changing.

In [8]:
fy24 = pull_year(2024)
agg24, agg25 = by_department(fy24), by_department(fy25)

depts = set(agg24) | set(agg25)
rows = []
for d in depts:
    a, b = agg24[d]["obligated"], agg25[d]["obligated"]
    delta_pct = (b - a) / a * 100 if a else float("inf")
    rows.append((d, a, b, b - a, delta_pct))

rows.sort(key=lambda r: r[2], reverse=True)  # by FY24 dollars

print(f"NAICS {NAICS} — FY24 → FY25 (between $1M and $5M deals)\n")
print(f"  {'FY24':>10}  {'FY25':>10}  {'Δ':>10}  {'Δ%':>7}  Department")
for d, a, b, dlt, pct in rows[:12]:
    pct_s = f"{pct:+.0f}%" if pct != float("inf") else "new"
    print(f"  ${a/1e6:>8.1f}M  ${b/1e6:>8.1f}M  ${dlt/1e6:>+8.1f}M  {pct_s:>7}  {d}")

NAICS 541512 — FY24 → FY25 (between $1M and $5M deals)

        FY24        FY25           Δ       Δ%  Department
  $   634.6M  $   781.3M  $  +146.7M     +23%  DEPT OF DEFENSE
  $   175.8M  $   123.6M  $   -52.2M     -30%  HEALTH AND HUMAN SERVICES, DEPARTMENT OF
  $   130.1M  $   114.4M  $   -15.7M     -12%  HOMELAND SECURITY, DEPARTMENT OF
  $   117.0M  $    86.6M  $   -30.3M     -26%  TREASURY, DEPARTMENT OF THE
  $    52.9M  $    59.7M  $    +6.8M     +13%  TRANSPORTATION, DEPARTMENT OF
  $    70.4M  $    40.2M  $   -30.2M     -43%  COMMERCE, DEPARTMENT OF
  $    12.0M  $    30.8M  $   +18.8M    +156%  GENERAL SERVICES ADMINISTRATION
  $    25.3M  $    30.5M  $    +5.2M     +21%  STATE, DEPARTMENT OF
  $    40.5M  $    28.2M  $   -12.3M     -30%  AGRICULTURE, DEPARTMENT OF
  $    21.8M  $    28.0M  $    +6.2M     +28%  INTERIOR, DEPARTMENT OF THE
  $    15.4M  $    19.5M  $    +4.2M     +27%  LABOR, DEPARTMENT OF
  $     9.8M  $    17.8M  $    +8.0M     +81%  ENERGY, DEPARTMENT OF

# Bonus: let's see who's been winning the contracts

We can also get a sense of the competition by looking at the recipients on the contracts.

In [9]:
def by_recipient(contracts):
    agg = defaultdict(lambda: {"obligated": 0.0, "count": 0, "average": 0.0})
    for c in contracts:
        rec = c.get("recipient") or {}
        label = f"{rec.get('legal_business_name')} ({rec.get('uei')})"
        agg[label]["obligated"] += float(c.get("obligated") or 0)
        agg[label]["count"] += 1
        agg[label]["average"] = agg[label]["obligated"] / agg[label]["count"]
    return agg


ranked_recipients = sorted(by_recipient(fy25).items(), key=lambda x: x[1]["obligated"], reverse=True)

print(f"Top 10 recipients of NAICS {NAICS} in FY25 (between $1M and $5M deals)\n")
print(f"  {'Dollars':>10}  {'Deals':>6}  {'Deal Ave':>8}  Recipient (UEI)")
for recipient, v in ranked_recipients[:10]:
    print(f"  ${v['obligated']/1e6:>8.1f}M    {v['count']:>3}   ${v['average']/1e6:>5.1f}M   {recipient}")

Top 10 recipients of NAICS 541512 in FY25 (between $1M and $5M deals)

     Dollars   Deals  Deal Ave  Recipient (UEI)
  $   126.2M     60   $  2.1M   LEIDOS, INC. (UE9QJD4KK1L6)
  $    56.7M     17   $  3.3M   JOHNSON CONTROLS BUILDING AUTOMATION SYSTEMS, LLC (LFD2YFMZJ1J3)
  $    44.7M     24   $  1.9M   HPI FEDERAL LLC (DJRUN4KK1HK3)
  $    30.4M      9   $  3.4M   GENERAL DYNAMICS INFORMATION TECHNOLOGY, INC. (SMNWM6HN79X5)
  $    28.8M     11   $  2.6M   DELOITTE CONSULTING LLP (CKV2L9GZKJK3)
  $    27.2M     13   $  2.1M   ACCENTURE FEDERAL SERVICES LLC (C47BNA8GM833)
  $    26.4M      9   $  2.9M   SIEMENS GOVERNMENT TECHNOLOGIES INC (R2VGJTEMCNL5)
  $    19.7M     13   $  1.5M   CAE USA INC. (HVQBAL5E7R23)
  $    18.7M     12   $  1.6M   CDW GOVERNMENT LLC (PHZDZ8SJ5CM1)
  $    17.4M      6   $  2.9M   EMC CORPORATION (ETMWNWTZ1CW5)
